# Logistic Regression

## Contents

- 1. Introduction
- 2. Dataset Overview
  - 2.1. Loading the Dataset
  - 2.2. Features and Target
  - 2.3. Image and Class Distribution
  - 2.4. Image Preprocessing
- 3. Loss Function
  - 3.1. Binary Cross-Entropy
- 4. Evaluation Metrics
  - 4.1. Confusion Matrix
  - 4.2. Accuracy
  - 4.3. Precision
  - 4.4. Recall
  - 4.5. F1-score
- 5. Logistic Regression with Gradient Descent
  - 5.1. Gradient of Binary Cross-Entropy
  - 5.2. NumPy Implementation
- 6. Logistic Regression with Scikit-learn
- 7. Logistic Regression with PyTorch
- 8. Comparison and Conclusions
  - 8.1. Evaluation Metrics
  - 8.2. Confusion Matrices
  - 8.3. Prediction Error Analysis
  - 8.4. Implementation Comparison
  - 8.5. Conclusions

## 1. Introduction

In the previous experiment, we considered **linear regression**, where the target is a continuous value and the model predicts

$$
z = X\mathbf{w} + b.
$$

For binary classification, the target instead belongs to one of two classes:

$$
y \in \{0, 1\}.
$$

The linear expression $z$ cannot be interpreted directly as a probability because

$$
z \in (-\infty, +\infty).
$$

To map it into the interval $(0, 1)$, logistic regression applies the **sigmoid function**:

$$
\sigma(z)
=
\frac{1}{1 + e^{-z}}.
$$

The model therefore predicts

$$
p
=
P(y=1 \mid X)
=
\sigma(X\mathbf{w}+b).
$$

Here, $p$ is a continuous probability rather than the final class prediction.

To obtain a class label, a decision threshold is applied. For the default threshold of $0.5$,

$$
\hat{y}
=
\begin{cases}
1, & p \geq 0.5, \\
0, & p < 0.5.
\end{cases}
$$

Thus, compared with linear regression, the linear part of the model remains the same:

$$
X\mathbf{w}+b.
$$

The main changes are:

1. The target is binary rather than continuous.
2. The linear output is transformed into a probability using the sigmoid function.
3. A classification loss is required.
4. Predictions are evaluated using classification metrics.

Although the model is called **logistic regression**, it is used here as a binary classifier.

## 2. Dataset Overview

The experiment uses the **Oxford-IIIT Pet Dataset** provided through `torchvision`.

The original dataset contains images of multiple cat and dog breeds. For this experiment, the breed information is ignored and the task is reduced to binary image classification:

$$
\text{image}
\longrightarrow
\{\text{Cat}, \text{Dog}\}.
$$

`torchvision.datasets.OxfordIIITPet` provides a binary target specifically for this task through

```python
target_types="binary-category"
```

with the following class mapping:

$$
0 = \text{Cat},
\qquad
1 = \text{Dog}.
$$

The **Dog** class is therefore treated as the positive class throughout the experiment.

### 2.1. Loading the Dataset

The dataset provides predefined `trainval` and `test` splits.

The training split is used for fitting model parameters, while the test split remains unseen during training and is used only for the final evaluation.

Using the predefined split also allows all implementations in this experiment to be trained and evaluated on exactly the same observations.


In [ ]:
from torchvision.datasets import OxfordIIITPet

train = OxfordIIITPet(
    "../../../artifacts/datasets/OxfordIIITPet",
    split="trainval",
    target_types="binary-category",
)

test = OxfordIIITPet(
    "../../../artifacts/datasets/OxfordIIITPet",
    split="test",
    target_types="binary-category",
)

### 2.2. Features and Target

Unlike the California Housing dataset used for linear regression, the input observations are now images rather than tabular feature vectors.

For one RGB image with height $H$ and width $W$,

$$
x \in \mathbb{R}^{3 \times H \times W}.
$$

Logistic regression expects each observation to be represented as a one-dimensional feature vector.

Therefore, after preprocessing, each image is flattened:

$$
3 \times H \times W
\longrightarrow
3HW.
$$

For a dataset containing $n$ images,

$$
X \in \mathbb{R}^{n \times 3HW},
\qquad
y \in {0,1}^{n}.
$$

Each pixel-channel value therefore becomes an individual input feature for the model.

This representation deliberately ignores the explicit two-dimensional structure of the image. Logistic regression receives only a vector of numerical features and does not directly model local spatial patterns.

### 2.3. Image and Class Distribution

Before training, we inspect several examples from both classes.

This serves two purposes:

* To verify that the images and labels were loaded correctly;
* To understand the visual variability of the classification problem.

We also inspect the number of Cat and Dog observations in each dataset split.

Class distribution is important for interpreting classification metrics. In a strongly imbalanced dataset, high accuracy may be achieved simply by favoring the majority class.

The class counts will therefore be checked before selecting the final evaluation metrics.

### 2.4. Image Preprocessing

The images in the dataset do not have a single fixed spatial size, while logistic regression requires every observation to contain the same number of features.

Therefore, all images are resized to a common resolution before being flattened.

Image values are also converted to floating-point numbers and scaled from the usual image range

$$
[0,255]
$$

to

$$
[0,1].
$$

Unlike the tabular features in the linear regression experiment, all pixel features already represent the same type of quantity and share the same numerical range after scaling.

For the initial experiment, we therefore use pixel scaling without additional StandardScaler standardization.

This keeps preprocessing simple and allows us to study logistic regression itself before introducing additional transformations.

If optimization proves difficult, feature or channel standardization can later be evaluated as a separate preprocessing experiment.

Further reading:

* [Torchvision transforms](https://docs.pytorch.org/vision/main/transforms.html)
* [Scikit-learn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
* [Importance of Feature Scaling](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_scaling_importance.html)

### 2.5. Preprocess dataset

The `OxfordIIITPet` dataset contains images with different spatial dimensions.

Logistic regression, however, requires every observation to contain the same number of features. Therefore, all images must first be transformed to a common size.

A direct resize to a fixed square resolution would change the original aspect ratio and distort the objects in the image. Instead, we use **letterbox preprocessing**:

$$
\text{Original image}
\rightarrow
\text{Resize preserving aspect ratio}
\rightarrow
\text{Padding}
\rightarrow
64 \times 64.
$$

The longest image side is resized to the target size while preserving the original aspect ratio. Padding is then added along the remaining dimension to obtain an image of exactly $64 \times 64$ pixels.

Since `OxfordIIITPet` returns PIL images while Albumentations operates on NumPy arrays, each image is first converted to `np.ndarray`.

Finally, pixel values are converted to `float32` and scaled from

$$
[0,255]
$$

to

$$
[0,1].
$$

In [ ]:
import numpy as np
from albumentations import Compose, LongestMaxSize, PadIfNeeded

IMAGE_SIZE = 64

transform = Compose([
    LongestMaxSize(max_size=IMAGE_SIZE),
    PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE),
])


def preprocess(image):
    image = np.asarray(image)
    image = transform(image=image)["image"]
    image = image.astype(np.float32) / 255.0
    return np.ravel(image)

Before applying preprocessing, we inspect one original image and its shape.

In [ ]:
import matplotlib.pyplot as plt

image, target = train[2]
image_np = np.asarray(image)

print(f"Shape: {image_np.shape}")
plt.imshow(image_np)
plt.title(f"Before preprocessing: {train.bin_classes[target]}")
plt.axis("off")
plt.show()

After preprocessing, the aspect ratio of the original image is preserved, while padding brings the image to the fixed target size.

In [ ]:
a = preprocess(image)
a.shape

In [ ]:
image_processed = preprocess(image).reshape(IMAGE_SIZE, IMAGE_SIZE, 3)

print(f"Shape: {image_processed.shape}")
plt.imshow(image_processed)
plt.title(f"After preprocessing: {train.bin_classes[target]}")
plt.axis("off")
plt.show()

After preprocessing, every image has the same shape:

$$
(64,64,3).
$$

The image can now be converted into the feature vector required by logistic regression:

$$
64 \times 64 \times 3
\rightarrow
12288.
$$

This flattening step will be performed when constructing the NumPy feature matrices rather than inside the image preprocessing function.

### 2.6. Load dataset

In [ ]:
x_train = np.stack([preprocess(image) for image, _ in train])
y_train = np.array([target for _, target in train], dtype=np.float32)

x_test = np.stack([preprocess(image) for image, _ in test])
y_test = np.array([target for _, target in test], dtype=np.float32)

In [ ]:
print(f"x_train shape: {x_train.shape}, y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}, y_test shape: {y_test.shape}")

in_features = len(x_train[0])
print(f"Features number: {in_features}")

## 3. Loss function

In linear regression, the model was trained by minimizing the difference between continuous predictions and continuous target values.

For binary classification, the model instead predicts a probability

$$
p_i = P(y_i=1 \mid x_i),
$$

where

$$
0 < p_i < 1.
$$

We therefore need a loss function that measures how well the predicted probability agrees with the true binary label.

### 3.1. Binary Cross-Entropy

For one observation, **Binary Cross-Entropy (BCE)** is defined as

$$
L(y,p)
=
-
\left[
y\log(p)
+
(1-y)\log(1-p)
\right].
$$

Because

$$
y \in \{0,1\},
$$

the expression reduces to one of two cases.

For a positive observation, $y=1$:

$$
L(1,p)
=
-\log(p).
$$

The loss becomes small when the model assigns a high probability to the positive class.

For a negative observation, $y=0$:

$$
L(0,p)
=
-\log(1-p).
$$

The loss becomes small when the model assigns a low probability to the positive class.

For $n$ observations, the mean binary cross-entropy is

$$
J(\mathbf{w},b)
=
-\frac{1}{n}
\sum_{i=1}^{n}
\left[
y_i\log(p_i)
+
(1-y_i)\log(1-p_i)
\right],
$$

where

$$
p_i
=
\sigma(x_i^T\mathbf{w}+b).
$$

The logistic regression problem can therefore be formulated as

$$
\min_{\mathbf{w},b}
J(\mathbf{w},b).
$$

Unlike least-squares linear regression, this optimization problem does not produce a general closed-form solution analogous to the normal equation.

The parameters are therefore estimated numerically.

## 4. Evaluation Metrics

Binary Cross-Entropy is used as the optimization objective, but a classification model should also be evaluated according to the classes it predicts.

For each observation, the predicted probability is converted into a class using a decision threshold.

For the default threshold

$$
t=0.5,
$$

the prediction is

$$
\hat{y}
=
\mathbb{1}[p \geq 0.5].
$$

The resulting predictions can be divided into four groups.

### 4.1. Confusion Matrix

Since the Dog class is defined as the positive class:

- **True Positive (TP)** — a Dog correctly predicted as Dog;
- **True Negative (TN)** — a Cat correctly predicted as Cat;
- **False Positive (FP)** — a Cat incorrectly predicted as Dog;
- **False Negative (FN)** — a Dog incorrectly predicted as Cat.

The confusion matrix can be written as

| | Predicted Cat | Predicted Dog |
|---|---:|---:|
| **Actual Cat** | TN | FP |
| **Actual Dog** | FN | TP |

The remaining classification metrics are calculated from these four values.

### 4.2. Accuracy

**Accuracy** measures the fraction of all observations classified correctly:

$$
\operatorname{Accuracy}
=
\frac{TP+TN}
{TP+TN+FP+FN}.
$$

Accuracy gives a useful overall measure when the classes are reasonably balanced.

However, it does not distinguish between false positives and false negatives.

### 4.3. Precision

**Precision** answers the question:

> Of all images predicted as Dog, how many are actually Dogs?

$$
\operatorname{Precision}
=
\frac{TP}{TP+FP}.
$$

Low precision means that many Cat images are incorrectly classified as Dogs.

### 4.4. Recall

**Recall** answers the question:

> Of all actual Dogs, how many did the model correctly identify?

$$
\operatorname{Recall}
=
\frac{TP}{TP+FN}.
$$

Low recall means that many Dogs are missed and classified as Cats.


### 4.5. F1-score

Precision and recall describe different types of classification error.

The **F1-score** combines them using their harmonic mean:

$$
F_1
=
2
\frac{
\operatorname{Precision}
\cdot
\operatorname{Recall}
}{
\operatorname{Precision}
+
\operatorname{Recall}
}.
$$

F1 becomes high only when both precision and recall are high.

For this experiment, we will compare the implementations using:

- Accuracy;
- Precision;
- Recall;
- F1-score;
- Confusion Matrix.

## 5. Logistic Regression with Gradient Descent

The optimization procedure itself is the same gradient descent algorithm used in the linear regression experiment:

$$
\theta_{t+1}
=
\theta_t
-
\eta
\nabla J(\theta_t).
$$

The difference is the objective function.

For logistic regression,

$$
z = X\mathbf{w}+b,
$$

$$
\mathbf{p}
=
\sigma(\mathbf{z}),
$$

and the model minimizes Binary Cross-Entropy.

Therefore, instead of introducing gradient descent again, we only need to derive the gradients of the new loss function.


### 5.1. Gradient of Binary Cross-Entropy

To train logistic regression with gradient descent, we need to calculate the gradient of the loss function with respect to the model parameters.

For a single observation, logistic regression consists of three consecutive operations:

$$
z = x^T\mathbf{w} + b,
$$

$$
p = \sigma(z),
$$

$$
L
=
-
\left[
y\log(p)
+
(1-y)\log(1-p)
\right].
$$

Therefore, the dependence of the loss on the weights can be written as

$$
\mathbf{w}
\longrightarrow
z
\longrightarrow
p
\longrightarrow
L.
$$

Using the chain rule,

$$
\frac{\partial L}{\partial \mathbf{w}}
=
\frac{\partial L}{\partial p}
\cdot
\frac{\partial p}{\partial z}
\cdot
\frac{\partial z}{\partial \mathbf{w}}.
$$

We can calculate these three derivatives separately.

#### 5.1.1. Derivative of BCE with respect to $p$

For one observation,

$$
L
=
-
\left[
y\log(p)
+
(1-y)\log(1-p)
\right].
$$

Differentiating with respect to $p$:

$$
\frac{\partial L}{\partial p}
=
-
\left[
\frac{y}{p}
+
(1-y)\frac{-1}{1-p}
\right].
$$

Therefore,

$$
\frac{\partial L}{\partial p}
=
\frac{1-y}{1-p}
-
\frac{y}{p}.
$$

#### 5.1.2. Derivative of the sigmoid with respect to $z$

The predicted probability is

$$
p
=
\sigma(z)
=
\frac{1}{1+e^{-z}}.
$$

The derivative of the sigmoid function is

$$
\frac{\partial p}{\partial z}
=
\sigma(z)(1-\sigma(z)).
$$

Since

$$
p=\sigma(z),
$$

we can write

$$
\frac{\partial p}{\partial z}
=
p(1-p).
$$

#### 5.1.3. Derivative of the linear function with respect to $\mathbf{w}$

For one observation,

$$
z=x^T\mathbf{w}+b.
$$

Therefore,

$$
\frac{\partial z}{\partial \mathbf{w}}
=
x^T.
$$

Now we combine all three derivatives using the chain rule:

$$
\frac{\partial L}{\partial \mathbf{w}}
=
\left(
\frac{1-y}{1-p}
-
\frac{y}{p}
\right)
p(1-p)x.
$$

Simplifying the first two factors:

$$
\left(
\frac{1-y}{1-p}
-
\frac{y}{p}
\right)
p(1-p)
$$

$$
=
p(1-y)-y(1-p),
$$

$$
=
p-py-y+yp,
$$

$$
=
p-y.
$$

Thus, for a single observation,

$$
\boxed{
\frac{\partial L}{\partial \mathbf{w}}
=
x(p-y)
}
$$

For the bias,

$$
\frac{\partial z}{\partial b}=1,
$$

so

$$
\boxed{
\frac{\partial L}{\partial b}
=
p-y
}
$$

For the whole dataset, Binary Cross-Entropy is averaged over $n$ observations:

$$
J(\mathbf{w},b)
=
\frac{1}{n}
\sum_{i=1}^{n}L_i.
$$

Combining the gradients for all observations gives the vectorized form

$$
\boxed{
\frac{\partial J}{\partial \mathbf{w}}
=
\frac{1}{n}
X^T(\mathbf{p}-\mathbf{y})
}
$$

and

$$
\boxed{
\frac{\partial J}{\partial b}
=
\frac{1}{n}
\sum_{i=1}^{n}(p_i-y_i)
}
$$

where

$$
\mathbf{p}
=
\sigma(X\mathbf{w}+b).
$$

The gradient descent update therefore becomes

$$
\mathbf{w}
\leftarrow
\mathbf{w}
-
\eta
\frac{\partial J}{\partial \mathbf{w}},
$$

$$
b
\leftarrow
b
-
\eta
\frac{\partial J}{\partial b}.
$$

An important result of the derivation is that the combination of **Binary Cross-Entropy and sigmoid** simplifies considerably:

$$
\frac{\partial L}{\partial z}
=
p-y.
$$

This is why the final gradient has a form similar to the gradient obtained for linear regression: it is again based on the difference between the model output and the true target.

### 5.2. NumPy Implementation

The NumPy implementation follows the mathematical formulation directly.

It contains the following main steps:

1. Calculate the linear logits.
2. Apply the sigmoid function.
3. Calculate Binary Cross-Entropy.
4. Calculate gradients.
5. Update the weights and bias with gradient descent.
6. Repeat until the selected number of iterations is reached.

The implementation is intentionally kept explicit so that each operation can be matched directly to the equations above.

This version serves as the reference implementation for understanding how logistic regression is trained internally.

#### 5.2.2 NumPy Implementation

In [ ]:
from implementations.gradient_descent import GradientDescentLogisticRegression

gd_regression = GradientDescentLogisticRegression(in_features=in_features)
gd_regression.fit(x_train, y_train, max_iter=100)
gd_y_pred = gd_regression.predict(x_test)

In [ ]:
gd_regression.bce_loss(y_test, gd_y_pred)